# 05 - Gradient boosting

Does a non-linear model beat the linear one from notebook 04? Same setup: train on 2022 and 2023, score on 2024, target `delta_s`, mean absolute error as the headline number.

I pick the model's settings by cross-validation inside the training seasons only, with folds grouped by race so laps from one race never end up on both sides of a split. 2024 isn't used to choose anything.

In [1]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from f1lab.features import prepare, split, mae, rmse
from f1lab.models import (GBM_FEATURES, gbm_features, linear_features, make_gbm, make_linear)

raw = pd.read_parquet("../data/laps.parquet")
clean = pd.read_parquet("../data/laps_clean.parquet")
train, val = split(prepare(raw, clean), "dev")
tracks = sorted(train["track"].unique())
X, y = gbm_features(train, tracks), train["delta_s"]
groups = train["season"].astype(str) + "-" + train["round"].astype(str)

## Choosing model size by grouped cross-validation (training seasons only)

In [2]:
rows = []
for leaves, trees in itertools.product([3, 4, 6, 8, 16, 31], [60, 100, 150, 200, 500]):
    scores = []
    for fit_idx, test_idx in GroupKFold(5).split(X, y, groups):
        m = make_gbm(leaves, trees).fit(X.iloc[fit_idx], y.iloc[fit_idx])
        scores.append(mae(y.iloc[test_idx], pd.Series(m.predict(X.iloc[test_idx]), index=y.index[test_idx])))
    rows.append((leaves, trees, np.mean(scores)))
cv = pd.DataFrame(rows, columns=["leaves", "trees", "cv MAE (s)"]).sort_values("cv MAE (s)").reset_index(drop=True)
best_leaves, best_trees = int(cv.loc[0, "leaves"]), int(cv.loc[0, "trees"])
print(f"chosen: {best_leaves} leaves, {best_trees} trees")
cv.head(5).round(4)

chosen: 8 leaves, 150 trees


,leaves,trees,cv MAE (s)
0,8,150,0.5610
1,8,200,0.5613
2,8,100,0.5630
3,6,200,0.5634
4,6,150,0.5637


In [3]:
# how does CV error move with model size?
cv.pivot(index="leaves", columns="trees", values="cv MAE (s)").round(3)

trees,60,100,150,200,500
leaves,,,,,
3,0.592,0.577,0.570,0.567,0.566
4,0.582,0.568,0.564,0.564,0.566
6,0.574,0.567,0.564,0.563,0.567
8,0.569,0.563,0.561,0.561,0.567
16,0.566,0.566,0.567,0.568,0.574
31,0.567,0.568,0.569,0.571,0.576


Mid-sized models win. Cross-validation error is lowest around 6 to 8 leaves and 100 to 200 trees (about 0.56). Tiny models underfit (3 leaves, 60 trees: 0.592) and large ones overfit (31 leaves, 500 trees: 0.576). With two seasons of data there's only so much structure to find before the model starts fitting noise from individual races.

## Validation on 2024, against everything so far

In [4]:
def gbm_val(columns=None, seeds=range(5)):
    cols = columns or GBM_FEATURES
    scores = []
    for seed in seeds:
        m = make_gbm(best_leaves, best_trees, seed).fit(gbm_features(train, tracks, cols), y)
        pred = pd.Series(m.predict(gbm_features(val, tracks, cols)), index=val.index)
        scores.append((mae(val["delta_s"], pred), rmse(val["delta_s"], pred)))
    return np.array(scores), m, pred

# the best baseline from notebook 03, recomputed here so it cannot go stale
bins = [0, 5, 10, 15, 20, 30, 100]
by_compound = train.groupby("compound")["delta_s"].mean()
by_age = train.assign(b=pd.cut(train["tyre_age"], bins)).groupby(["compound", "b"], observed=True)["delta_s"].mean()
vb = val.assign(b=pd.cut(val["tyre_age"], bins))
baseline_pred = pd.Series([by_age.get((c, b), np.nan) for c, b in zip(vb["compound"], vb["b"])], index=val.index)
baseline_mae = mae(val["delta_s"], baseline_pred.fillna(val["compound"].map(by_compound)))

scores, gbm, gbm_pred = gbm_val()
lin = make_linear().fit(linear_features(train), y)
lin_pred = pd.Series(lin.predict(linear_features(val)), index=val.index)
summary = pd.DataFrame(
    {
        "MAE (s)": {
            "best baseline (compound + tyre-age bucket)": baseline_mae,
            "linear, Huber (notebook 04)": mae(val["delta_s"], lin_pred),
            "gradient boosting (mean of 5 seeds)": scores[:, 0].mean(),
        }
    }
).round(3)
print(f"gradient boosting MAE by seed: {scores[:, 0].round(3)} (spread {scores[:, 0].std():.4f})")
summary

gradient boosting MAE by seed: [0.598 0.6   0.6   0.6   0.598] (spread 0.0008)


,MAE (s)
best baseline (compound + tyre-age bucket),0.855
"linear, Huber (notebook 04)",0.629
gradient boosting (mean of 5 seeds),0.599


Gradient boosting reaches 0.599 s, against 0.629 for the linear model and 0.855 for the best baseline. The five seeds agree to within 0.002, so the 0.030 s edge over the linear model (about 5%) is real. Most of the gain over the baselines still comes from the fuel term, but the non-linear model adds a fair bit on top.

## Which inputs matter? (ablations, 5 seeds each)

In [5]:
def without(*drop):
    return [c for c in GBM_FEATURES if c not in drop]

variants = {
    "all features": GBM_FEATURES,
    "no track": without("track"),
    "no temperatures": without("track_temp", "air_temp"),
    "no compound": without("compound"),
    "no tyre age": without("tyre_age"),
    "no fuel features": without("fuel_frac", "laps_remaining", "lap_number"),
}
ablation = pd.Series({name: gbm_val(cols)[0][:, 0].mean() for name, cols in variants.items()}, name="MAE (s)")
pd.DataFrame({"MAE (s)": ablation, "change vs all features": ablation - ablation["all features"]}).round(3)

,MAE (s),change vs all features
all features,0.599,0.000
no track,0.605,0.006
no temperatures,0.604,0.005
no compound,0.599,-0.000
no tyre age,0.653,0.054
no fuel features,0.870,0.271


That ablation ran on 2024, so using it to decide which inputs to keep would make 2024 a slightly optimistic score. A choice like that should be made on the training seasons. Here's the same comparison by grouped cross-validation, without touching 2024:

In [6]:
def cv_mae(columns):
    Xc = gbm_features(train, tracks, columns)
    scores = []
    for seed in range(3):
        for fit_idx, test_idx in GroupKFold(5).split(Xc, y, groups):
            m = make_gbm(best_leaves, best_trees, seed).fit(Xc.iloc[fit_idx], y.iloc[fit_idx])
            scores.append(mae(y.iloc[test_idx], pd.Series(m.predict(Xc.iloc[test_idx]), index=y.index[test_idx])))
    return np.mean(scores)

cv_ablation = pd.Series({name: cv_mae(cols) for name, cols in variants.items()}, name="CV MAE (s)")
pd.DataFrame({"CV MAE (s)": cv_ablation, "change vs all features": cv_ablation - cv_ablation["all features"]}).round(3)

,CV MAE (s),change vs all features
all features,0.563,0.000
no track,0.571,0.008
no temperatures,0.560,-0.003
no compound,0.563,0.001
no tyre age,0.617,0.054
no fuel features,0.902,0.340


In cross-validation only two inputs clearly matter: tyre age (+0.054 s when removed) and the fuel features (+0.340 s). Track is a small but steady help, costing +0.008 s when removed in cross-validation and +0.006 s on 2024. Compound makes no difference (+0.001 and 0.000). Temperatures are inconclusive: removing them is slightly better in cross-validation (-0.003) and slightly worse on 2024 (+0.005). I kept the full set of inputs, because nothing here justifies dropping any and I didn't want to lean on the 2024 numbers.

In [7]:
imp = pd.Series(gbm.booster_.feature_importance("gain"), index=GBM_FEATURES)
(imp / imp.sum()).sort_values(ascending=False).round(3)

fuel_frac         0.496
lap_number        0.169
track             0.135
tyre_age          0.115
laps_remaining    0.042
track_temp        0.021
air_temp          0.015
compound          0.005
race_laps         0.003
dtype: float64

Gain importance puts the track third (0.135), and removing it does cost a little (0.006 to 0.008 s), so importance and benefit roughly agree here. Fuel dominates: the fraction of the race remaining alone is about half of the total.

## Where the errors are

In [8]:
err = (val["delta_s"] - gbm_pred).abs()
stage = pd.cut(1 - val["laps_remaining"] / val["race_laps"], [0, .25, .5, .75, 1])
lin_err = (val["delta_s"] - lin_pred).abs()
pd.DataFrame({"gradient boosting": err.groupby(stage, observed=True).mean(), "linear": lin_err.groupby(stage, observed=True).mean()}).round(3)

,gradient boosting,linear
"(0.0, 0.25]",0.636,0.651
"(0.25, 0.5]",0.543,0.529
"(0.5, 0.75]",0.473,0.528
"(0.75, 1.0]",0.732,0.798


Gradient boosting is better in three of the four race stages, most clearly in the third quarter (0.473 against 0.528) and the last (0.732 against 0.798). The linear model is slightly better in the second quarter (0.529 against 0.543). I haven't tested why.

In [9]:
unseen = ~val["track"].isin(train["track"].unique())
print("tracks with no training data:", sorted(val.loc[unseen, "track"].unique()), f"({unseen.sum()} laps)")
print(f"unseen tracks: GBM {err[unseen].mean():.3f}, linear {lin_err[unseen].mean():.3f}")
print(f"seen tracks:   GBM {err[~unseen].mean():.3f}, linear {lin_err[~unseen].mean():.3f}")
print(val[unseen].assign(e=err[unseen]).groupby("track")["e"].mean().round(3).to_dict())

tracks with no training data: ['Imola', 'Monte Carlo', 'Shanghai'] (3130 laps)
unseen tracks: GBM 0.851, linear 0.953
seen tracks:   GBM 0.553, linear 0.571
{'Imola': 0.598, 'Monte Carlo': 1.066, 'Shanghai': 0.902}


Three 2024 tracks have no training data: Imola, Monte Carlo and Shanghai. Imola and Monaco's earlier races were wet and got excluded. On those tracks gradient boosting copes much better than the linear model (0.851 against 0.953), though both are far worse than at tracks they've seen (0.553 and 0.571). Monte Carlo is the hardest by a distance (1.066), a slow, narrow track unlike the rest.

## Where that leaves me

- Fuel is the big win. Tyre age is the only other input that clearly helps, and track adds a little.
- Gradient boosting beats the linear model by about 5% (0.599 against 0.629 s) and the best baseline by about 30% (0.855 s). That's worth the extra complexity.
- Unseen tracks are the weak spot. A non-linear model copes better than a linear one, but error is still much higher than at tracks it has seen.
- The remaining 0.6 s is partly unseen-track difficulty and partly noise the inputs can't explain, like traffic and incidents. A calibrated range is more useful than squeezing the point estimate further, so that's next.